# Building BO: Oracle Functions

This notebook focuses on the idea of the **oracle function**.
The oracle acts as a fast, computational surrogate for an expensive real experiment, allowing us to simulate optimisation campaigns without physically running reactions.

Instead of sending conditions to a robot or laboratory, Bayesian Optimisation will query the oracle, which instantly returns a predicted reaction outcome such as yield.
That makes it possible to:

- Rapidly prototype optimisation strategies
- Compare different kernels and acquisition functions
- Benchmark convergence behaviour
- Stress-test workflows before deploying them in the lab

Without the use of an oracle, the optimizer will suggest an experiment with conditions that neither exist in our data, nor can we obtain output data for it.

The oracle-building code lives in `scripts/common.py`.

**Note: the oracles for these workbooks have already been built, so this section is used mainly to explore how they function, are interacted with, and could be built, though there is no need to build your own**


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from scripts.common import build_oracle_from_spec, make_problem_spec
from scripts.oracle_generator import normalize_module_name, write_oracle_module

DATA_DIR = Path("data")


def show_oracle_report(model_names, performance):
    print("Selected oracle model(s):")
    if isinstance(model_names, dict):
        for target, name in model_names.items():
            print(f"  {target}: {name}")
    else:
        print(model_names)

    print("\nCross-validated performance:")
    for target, perf in performance.items():
        print(f"\nTarget: {target}")
        display(perf)


## Case study 1: Dataset_A1_Aspuru-Guzik_Suzuki_Numeric (numeric-only)

We load a small numeric dataset, define the feature and target columns, then train an oracle using the code from `scripts/common.py`.


In [ ]:
data_path = DATA_DIR / "dataset_a1_aspuru_guzik_suzuki_numeric.csv"
df = pd.read_csv(data_path)

spec = make_problem_spec(
    df,
    features=[
        "temperature_C",
        "Pd_mol_percent",
        "ArBpin_equiv",
        "K3PO4_equiv",
    ],
    targets=["yield_percent"],
    continuous=[
        "temperature_C",
        "Pd_mol_percent",
        "ArBpin_equiv",
        "K3PO4_equiv",
    ],
    categorical=[],
)

oracle, models, model_names, performance = build_oracle_from_spec(df=df, spec=spec)
show_oracle_report(model_names, performance)

x_test = [85.0, 1.5, 1.3, 2.0]
print("\nOracle output:")
print(oracle(x_test))


## Case study 2: Dataset_B4_Jensen_Case_IV (categorical + multi-target)

In this example we include one categorical feature and two targets. The core pattern is still the same:
load the table, define the feature/target metadata, then build the oracle from that specification.


In [ ]:
data_path = DATA_DIR / "dataset_b4_jensen_case_iv.csv"
df = pd.read_csv(data_path)

spec = make_problem_spec(
    df,
    features=[
        "ligand",
        "res_time",
        "temperature",
        "catalyst_loading",
    ],
    targets=["yield", "turnover"],
    continuous=[
        "res_time",
        "temperature",
        "catalyst_loading",
    ],
    categorical=["ligand"],
)

oracle, models, model_names, performance = build_oracle_from_spec(df=df, spec=spec)
show_oracle_report(model_names, performance)

x_test = [spec.categorical_levels["ligand"][0], 300.0, 90.0, 1.0]
print("\nOracle outputs:")
print(oracle(x_test))


### From notebook pattern to reusable oracle

The two case studies above already show the core structure of an oracle:

1. Load a dataset.
2. Define the input columns.
3. Define the target columns.
4. Fit a model from that specification.
5. Wrap the fitted model in an `oracle(x)` function.

In workbooks 4 and 5 you can load a prebuilt oracle from `scripts/` rather than rebuilding one inside the notebook. However, instructions are given below if you want to build a new oracle for a custom dataset to use for benchmarking.


### Optional: Create a new oracle script

Fill in the lists below for your own dataset. If you set `WRITE_ORACLE_SCRIPT = True` the cell writes a new dataset module into `scripts/`.

This assumes your CSV is already in `data/` and has sensible headers. After writing the module, run the final cell to load it and query it in the same style as case study 2. If you want workbook 5 to list the new oracle by key, you still need to add it to `scripts/registry.py` afterwards.


In [ ]:
# ===== Fill in these lines for your dataset =====
NEW_MODULE_NAME = "dataset_new_reaction"  # Required: new script name, e.g. dataset_g1_my_reaction
NEW_DATA_FILENAME = "my_dataset.csv"  # Required: CSV filename in data/
NEW_FEATURES = ["temperature_c", "res_time_min", "ligand"]  # Required: input columns used by the oracle
NEW_TARGETS = ["yield_pct"]  # Required: output column(s) predicted by the oracle
NEW_CONTINUOUS = ["temperature_c", "res_time_min"]  # Required: subset of NEW_FEATURES that are numeric
NEW_CATEGORICAL = ["ligand"]  # Required: subset of NEW_FEATURES that are categorical
NEW_DEFAULT_TARGET = "yield_pct"  # Required: one target from NEW_TARGETS for workbook 5 defaults

# ===== Usually leave these alone until you are ready to write the file =====
WRITE_ORACLE_SCRIPT = False  # Change to True to write the new oracle module into scripts/
OVERWRITE_ORACLE_SCRIPT = False  # Change to True only if you want to replace an existing module

if WRITE_ORACLE_SCRIPT:
    module_path = write_oracle_module(
        module_name=NEW_MODULE_NAME,
        data_filename=NEW_DATA_FILENAME,
        features=NEW_FEATURES,
        targets=NEW_TARGETS,
        continuous=NEW_CONTINUOUS,
        categorical=NEW_CATEGORICAL,
        default_target=NEW_DEFAULT_TARGET,
        output_dir="scripts",
        overwrite=OVERWRITE_ORACLE_SCRIPT,
    )
    print(f"\nWrote oracle module: {module_path}")
    print("Run the next cell to load the new module and query it with an example row.")
    print("Add it to scripts/registry.py if you want workbook 5 to discover it by key.")
else:
    print("\nWrite step skipped. Set WRITE_ORACLE_SCRIPT = True when you are ready to create the module in scripts/.")


### Optional: Test and query the new oracle

Run this after the previous cell has written your module. It loads the new script from `scripts/`, builds the oracle, shows the selected model(s), and queries the oracle using the first row of your CSV.


In [ ]:
if not WRITE_ORACLE_SCRIPT:
    print("Custom-oracle test skipped. Enable WRITE_ORACLE_SCRIPT and create your module to run this optional exercise.")
else:
    import importlib
    import sys

    normalized_module_name = normalize_module_name(NEW_MODULE_NAME)
    module_path = Path("scripts") / f"{normalized_module_name}.py"
    if not module_path.exists():
        raise FileNotFoundError(
            f"{module_path} does not exist yet. Set WRITE_ORACLE_SCRIPT = True in the previous cell and run it first."
        )

    module_import_path = f"scripts.{normalized_module_name}"
    importlib.invalidate_caches()
    if module_import_path in sys.modules:
        new_oracle_module = importlib.reload(sys.modules[module_import_path])
    else:
        new_oracle_module = importlib.import_module(module_import_path)

    bundle = new_oracle_module.build_oracle()
    show_oracle_report(bundle.model_names, bundle.performance)

    example_df = new_oracle_module.load_dataset()
    example_row = example_df.iloc[0]
    x_test = [example_row[column] for column in bundle.spec.features]

    print("\nFeature order:")
    print(bundle.spec.features)
    print("\nExample query:")
    print(x_test)
    print("\nOracle outputs:")
    print(bundle.oracle(x_test))
